In [0]:
%sql
use catalog investment_pyspark;
create schema if not exists Silver;

In [0]:
df_bronze = spark.read.table("investment_pyspark.bronze.holdings_raw")
df_bronze.show()

In [0]:
df_bronze.printSchema()

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import DecimalType, IntegerType

In [0]:
df_silver = (df_bronze
  .withColumn("Instrument", F.upper(F.trim(F.col("Instrument"))))
  .withColumn("Qty", F.col("Qty").cast(IntegerType()))
  .withColumn("Avg_cost", F.col("Avg_cost").cast(DecimalType(18,2)))
  .withColumn("Current_price", F.col("LTP").cast(DecimalType(18,2)))
  .withColumn("Invested_value", F.col("Invested").cast(DecimalType(18,2)))
  .withColumn("Current_value",F.col("Cur_val").cast(DecimalType(18,2)))
  .withColumn("Unrealized_pnl", F.col("P_L").cast(DecimalType(18,2)))
  .withColumn("Net_change_pct", F.col("Net_chg").cast(DecimalType(18,2)))
  .withColumn("Day_change_pct", F.col("Day_chg").cast(DecimalType(18,2)))
  .withColumn(
      "assent_class", 
              F.when(
                  F.col("Instrument").rlike("^[0-9]")
                     | F.col("Instrument").rlike("(BOND|SGB|SGL|MML|NFL|KFL)"),
                             "DEBT")
                     .when(F.col("Instrument").rlike("(BEES|ETF|MON100)"),"ETF")
                     .otherwise("EQUITY"))
  .withColumn("silver_processed_timestamp", F.current_timestamp())
  .filter(F.col("Instrument").isNotNull()
                      & (F.col("Qty")>0)
                      & (F.col("Invested_value")>0))
  .select("Instrument","assent_class","Qty","Avg_cost","Current_price","Invested_value","Current_value","Unrealized_pnl","Net_change_pct","Day_change_pct","silver_processed_timestamp")
)

In [0]:
#display(df_silver)
df_silver.printSchema()


In [0]:
df_silver.write.format("delta").mode("overwrite").saveAsTable("investment_pyspark.silver.holdings_clean")

In [0]:
%sql
select * from investment_pyspark.silver.holdings_clean;